# Kaggle setup and interface checks

Publish the project source as a Kaggle dataset, increasing the version in `pyproject.toml` before every dataset update. Attach the same source dataset version for both teammates. Run this notebook directly in Kaggle or with its remote kernel in VS Code. Scripts run through `%run`; outputs belong under `/kaggle/working/`. Restart the kernel after changing the attached source version.


In [ ]:
from pathlib import Path
import os
import sys
import tomllib

# Edit the source dataset path or set ALSEG_PROJECT_DIR.
project_dir = Path(os.environ.get(
    "ALSEG_PROJECT_DIR", "/kaggle/input/datasets/YOUR_ACCOUNT/alseg-source"
))
if not (project_dir / "alseg" / "core" / "selection.py").is_file():
    raise FileNotFoundError(f"Attach the source dataset and set project_dir: {project_dir}")
sys.path.insert(0, str(project_dir))
config_path = project_dir / "configs" / "base.toml"
setup_script = project_dir / "setup_project.py"
tests_dir = project_dir / "tests"
fixture_script = tests_dir / "synthetic.py"
with (project_dir / "pyproject.toml").open("rb") as file:
    project_version = tomllib.load(file)["project"]["version"]
print("Project:", project_dir, "Version:", project_version)


In [ ]:
%run "$setup_script" --config "$config_path" --inspect-data


In [ ]:
%run -m pytest -- -q -o cache_dir=/kaggle/working/alseg/pytest-cache "$tests_dir"


In [ ]:
%run "$fixture_script"
from alseg.core.selection import validate_selection

for dimension in (2, 5):
    fixture = make_fixture(dimension)
    validate_selection(fixture.request, fixture.expected_top_uncertainty)
    print("Embedding shape:", tuple(fixture.request.embeddings.shape))
    print("Expected top-uncertainty IDs (tie-break by ID):", fixture.expected_top_uncertainty)

# Person 2 implements Selector.select(request) in adapters/acquisition.py.
# validate_selection checks output validity; algorithm tests check correctness.


In [ ]:
import importlib.metadata
import json
import platform
import torch

setup_dir = Path("/kaggle/working/alseg/setup")
setup_dir.mkdir(parents=True, exist_ok=True)
environment = {
    "project_version": project_version,
    "source_dir": str(project_dir),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
(setup_dir / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
packages = sorted({
    f"{dist.metadata['Name']}=={dist.version}"
    for dist in importlib.metadata.distributions() if dist.metadata['Name']
})
(setup_dir / "requirements-kaggle.txt").write_text("\n".join(packages) + "\n")
print(json.dumps(environment, indent=2))
print("Environment snapshot:", setup_dir)


In [ ]:
import uuid
import numpy as np
from PIL import Image
from IPython.display import display

from alseg.adapters.voc import VocImages, VocEvaluation, VocOracle
from alseg.data import find_voc_root, load_manifest

with config_path.open("rb") as file:
    data_config = tomllib.load(file)["data"]
explicit_root = data_config["voc_root"]
voc_root = Path(explicit_root) if explicit_root else find_voc_root(Path(data_config["dataset_root"]))
manifest = load_manifest(voc_root)
pool = VocImages(voc_root, manifest)
evaluation = VocEvaluation(voc_root, manifest)
# This diagnostic query is a separate preview run, never part of benchmark L.
preview_dir = Path("/kaggle/working/alseg/previews") / uuid.uuid4().hex
oracle = VocOracle(voc_root, manifest, preview_dir / "oracle_queries.jsonl")
image = pool.get_image(0)
mask = oracle.query([0])[0]
colors = np.array([(i * 37 % 256, i * 67 % 256, i * 97 % 256) for i in range(256)], dtype=np.uint8)
colors[255] = [255, 0, 255]  # Void, ignored during training/evaluation.
rgb = Image.fromarray(image.permute(1, 2, 0).numpy())
colored_mask = Image.fromarray(colors[mask.numpy()])
print("Train ID 0:", manifest.pool_ids[0], "classes:", mask.unique().tolist())
print("RGB | class mask (void=magenta) | overlay")
preview = Image.new("RGB", (rgb.width * 3, rgb.height))
for index, panel in enumerate((rgb, colored_mask, Image.blend(rgb, colored_mask, 0.5))):
    preview.paste(panel, (index * rgb.width, 0))
display(preview)
print("Val ID 0:", manifest.val_ids[0], "classes:", evaluation.get_mask(0).unique().tolist())
print("Preview query audit:", preview_dir / "oracle_queries.jsonl")


## Next steps

Implement acquisition adapters against `alseg/interfaces.py` and the core models. Implement the segmentation/Random pipeline separately. Before training, enable a suitable GPU, audit two checkpoints, and freeze the recipe. Data access and the oracle are implemented. Encoder, segmentation training, acquisition algorithms, and the benchmark are next.
